# 00 — Canonical data and Evaluation Protocol v1.0

Recommended runtime: T4 High-RAM, CPU computation only. Prerequisites: the three existing CSVs in Drive `original data/`. Full mode preserves every purchase line. Smoke mode selects a fixed hash-ordered customer subset, retains all of their dated purchases, and limits diagnostic queries per anchor. No globally warm-user filter is applied. Outputs: typed canonical Parquet, stable sorted ID maps, C0, 15 anchors, partitioned query rosters and unique labels.

Run only in your own Colab session. This notebook mounts Drive and runs the selected pipeline stage. `MODE="smoke"` is the default; smoke outputs are diagnostics, not full H&M results. Full execution has not been run by the authoring agent. Existing source CSVs and CLIP artifacts are read-only. Metadata/images are static snapshots, not proven historical availability.

## Native Colab dependencies
Run in a fresh runtime. A resolver conflict is an error to investigate; do not remove the PyTorch constraints or reinstall CUDA packages to hide it.

In [1]:
import importlib.metadata
import subprocess
import sys
import tempfile

# Use Colab native Python. Never install the repository's uv environment here.
# Pin any installed PyTorch packages so pip cannot replace Colab CUDA builds.
constraints = []
for package in ("torch", "torchvision", "torchaudio"):
    try:
        constraints.append(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        pass
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt") as handle:
    handle.write("\n".join(constraints))
    handle.flush()
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--constraint",
            handle.name,
            *["duckdb==1.5.6", "pyarrow>=18,<27"],
        ],
        check=True,
    )

## Shared protocol and validation contracts
Small helpers are repeated so this notebook has no repository-module dependency.

In [2]:
import hashlib
import json
import math
from datetime import date, timedelta

import numpy as np
import pyarrow as pa

PROTOCOL_VERSION = "1.0"
ARTIFACT_VERSION = "hm-mvp-1.0.0"
FEATURE_SCHEMA_VERSION = "ranking-1.0"
SEED = 20261031
CUTOFF = date(2020, 7, 28)
CHANNELS = ("popularity", "itemcf", "als", "two_tower", "text", "image")
RRF_K0 = 60
CANDIDATES_PER_CHANNEL = 100
FUSED_CANDIDATE_BUDGET = 500
CANDIDATE_SCHEMA = pa.schema(
    [
        ("query_id", pa.string()),
        ("customer_id", pa.string()),
        ("anchor_date", pa.date32()),
        ("article_id", pa.string()),
        ("channel", pa.string()),
        ("raw_score", pa.float32()),
        ("source_rank", pa.int32()),
        ("retriever_version", pa.string()),
    ]
)


def require(condition, message):
    if not condition:
        raise ValueError(message)


def identity(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode()).hexdigest()


def anchor_schedule():
    starts = [
        ("ranking_train", CUTOFF),
        ("validation", date(2020, 8, 16)),
        ("test", date(2020, 9, 4)),
    ]
    return [
        {
            "split": split,
            "anchor_date": start + timedelta(days=offset),
            "window_end": start + timedelta(days=offset + 7),
        }
        for split, start in starts
        for offset in (0, 3, 6, 9, 12)
    ]


def unique_ranked(items):
    return list(dict.fromkeys(items))


def query_metrics(predictions, truth, k=20):
    truth = set(truth)
    require(bool(truth), "Empty ground truth is not an eligible metric denominator.")
    predictions = unique_ranked(predictions)[:k]
    hits = [int(item in truth) for item in predictions]
    recall = sum(hits) / len(truth)
    dcg = sum(hit / math.log2(j + 2) for j, hit in enumerate(hits))
    ideal = sum(1 / math.log2(j + 2) for j in range(min(k, len(truth))))
    cumulative, precision_sum = 0, 0.0
    for j, hit in enumerate(hits, 1):
        cumulative += hit
        precision_sum += hit * cumulative / j
    return {
        "recall": recall,
        "ndcg": dcg / ideal,
        "ap": precision_sum / min(k, len(truth)),
    }


def stable_top(items, scores, limit=100):
    best = {}
    for item, score in zip(items, scores, strict=True):
        score = float(score)
        require(math.isfinite(score), "Nonfinite retrieval score.")
        best[item] = max(best.get(item, -math.inf), score)
    return sorted(best.items(), key=lambda pair: (-pair[1], pair[0]))[:limit]


def fuse_query(rows, excluded=None, budget=500):
    by_item = {}
    keys = set()
    for row in rows:
        if row["channel"] == excluded:
            continue
        key = row["channel"], row["article_id"]
        require(key not in keys, "Duplicate source candidate before fusion.")
        keys.add(key)
        entry = by_item.setdefault(
            row["article_id"],
            {"article_id": row["article_id"], "rrf": 0.0, "sources": {}},
        )
        entry["sources"][row["channel"]] = (row["source_rank"], row["raw_score"])
        entry["rrf"] += 1.0 / (RRF_K0 + row["source_rank"])
    ordered = sorted(by_item.values(), key=lambda row: (-row["rrf"], row["article_id"]))
    return ordered[:budget], len(ordered)


def validate_candidates(table, roster, catalog, channel):
    require(table.schema.equals(CANDIDATE_SCHEMA), "Candidate schema mismatch.")
    query_lookup = {r["query_id"]: r for r in roster}
    seen, groups = set(), {}
    for row in table.to_pylist():
        qid, item = row["query_id"], row["article_id"]
        require(qid in query_lookup, "Candidate references an unknown query.")
        q = query_lookup[qid]
        require(
            row["customer_id"] == q["customer_id"]
            and row["anchor_date"] == q["anchor_date"],
            "Query identity mismatch.",
        )
        require(item in catalog and isinstance(item, str), "Ineligible candidate ID.")
        require(row["channel"] == channel, "Wrong channel in candidate shard.")
        require(
            row["retriever_version"] == ARTIFACT_VERSION, "Retriever version mismatch."
        )
        require((qid, item) not in seen, "Duplicate query/article candidate.")
        require(math.isfinite(row["raw_score"]), "Nonfinite candidate score.")
        seen.add((qid, item))
        groups.setdefault(qid, []).append(row)
    for rows in groups.values():
        rows.sort(key=lambda r: r["source_rank"])
        require(len(rows) <= 100, "More than 100 candidates for a channel.")
        require(
            [r["source_rank"] for r in rows] == list(range(1, len(rows) + 1)),
            "Source ranks are not contiguous from one.",
        )
        expected = stable_top(
            [r["article_id"] for r in rows], [r["raw_score"] for r in rows]
        )
        require(
            [r["article_id"] for r in rows] == [i for i, _ in expected],
            "Unstable score/article ordering.",
        )
    require(
        set(groups) == set(query_lookup), "A query silently disappeared from a channel."
    )


def validate_mapping(ids, rows):
    require(len(ids) == len(rows), "Mapping length mismatch.")
    require(len(ids) == len(set(ids)), "Duplicate article ID in mapping.")
    require(list(rows) == list(range(len(ids))), "Noncontiguous embedding rows.")
    require(
        all(isinstance(x, str) and len(x) == 10 and x.isdigit() for x in ids),
        "Article IDs must preserve ten-digit strings.",
    )


def validate_manifest(manifest, mode, catalog_id=None):
    fields = {
        "artifact_version",
        "protocol_version",
        "mode",
        "input_paths",
        "input_fingerprints",
        "model_parameters",
        "random_seed",
        "row_count",
        "snapshot_or_anchor",
        "catalog_id",
        "feature_schema_version",
        "output_paths",
        "validation_status",
        "files",
        "stage",
    }
    require(fields <= manifest.keys(), "Incomplete artifact manifest.")
    from pathlib import PurePosixPath

    require(
        set(manifest["files"]) == set(manifest["output_paths"]),
        "Output manifest mismatch.",
    )
    for name in manifest["files"]:
        path = PurePosixPath(name)
        require(
            not path.is_absolute() and ".." not in path.parts and "\\" not in name,
            "Unsafe artifact-relative path.",
        )
    require(
        manifest["artifact_version"] == ARTIFACT_VERSION
        and manifest["protocol_version"] == PROTOCOL_VERSION,
        "Artifact/protocol mismatch.",
    )
    require(manifest["mode"] == mode, "Smoke/full artifact mixing is forbidden.")
    require(manifest["validation_status"] == "validated", "Artifact is not complete.")
    if catalog_id is not None:
        require(manifest["catalog_id"] == catalog_id, "Catalog identity mismatch.")


def metric_smoke_checks():
    result = query_metrics(["a", "x", "b"], {"a", "b"}, 3)
    require(result["recall"] == 1, "Recall formula failed.")
    require(abs(result["ap"] - (1 + 2 / 3) / 2) < 1e-12, "AP formula failed.")
    require(query_metrics([], {"a"})["ndcg"] == 0, "Missing recommendations lost.")
    require(
        query_metrics(["a", "a"], {"a", "b"}, 2)["recall"] == 0.5,
        "Duplicate recommendation received extra credit.",
    )
    schedule = anchor_schedule()
    require(len(schedule) == 15, "Anchor count mismatch.")
    require(schedule[-1]["window_end"] == date(2020, 9, 23), "Test overflow.")

## Runtime, storage and one execution mode
Change only `MODE` between `smoke` and `full`. Outputs use `<stage>/mvp_v1/<mode>/` beneath the specified Drive data root. Completion manifests are written last. Changed inputs or parameters require a new artifact version; completed runs are immutable. Local disk checks include a reserve, but cannot guarantee Colab or Drive quota.

In [3]:
import os
import shutil
import time
from pathlib import Path

import duckdb

DRIVE_ROOT = Path("/content/drive/MyDrive/projects/e_commerce_recsys/data")
LOCAL_ROOT = Path("/content/data/hm")
WORK_ROOT = Path("/content/recsys_work")
MODE = "full"  # 'smoke' or 'full'; smoke results are diagnostics only.
QUERY_BATCH_SIZE = 256
SMOKE_CUSTOMERS = 2000
SMOKE_QUERIES_PER_ANCHOR = 32
MAX_HISTORY_ITEMS = 100  # Most recent distinct items; query representation only.
MEMORY_LIMIT = "4GB"
THREADS = 2
RUN_NAMESPACE = "mvp_v1"


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b""):
            h.update(chunk)
    return h.hexdigest()


def disk_guard(path, needed_bytes, operation):
    root = Path(path)
    while not root.exists():
        root = root.parent
    free = shutil.disk_usage(root).free
    require(
        free >= needed_bytes + 2 * 1024**3,
        f"{operation}: need {needed_bytes / 1024**3:.2f} GiB plus 2 GiB reserve; "
        f"only {free / 1024**3:.2f} GiB free at {root}. No population truncated.",
    )


def copy_verified(source, target, expected=None):
    source, target = Path(source), Path(target)
    require(source.is_file(), f"Missing prerequisite file: {source}")
    digest = expected or sha256_file(source)
    if target.is_file() and target.stat().st_size == source.stat().st_size:
        if sha256_file(target) == digest:
            return target
    disk_guard(target.parent, source.stat().st_size, "Stage input")
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + ".partial")
    shutil.copyfile(source, partial)
    require(sha256_file(partial) == digest, "Staged file checksum mismatch.")
    os.replace(partial, target)
    return target


def stage_path(stage, root):
    return Path(root) / stage / RUN_NAMESPACE / MODE


def load_stage(stage, catalog_id=None, wanted=None):
    remote = stage_path(stage, DRIVE_ROOT)
    marker = remote / "_COMPLETE.json"
    require(marker.is_file(), f"Complete prerequisite missing: {marker}")
    manifest = json.loads(marker.read_text())
    validate_manifest(manifest, MODE, catalog_id)
    require(manifest["stage"] == stage, "Wrong stage manifest.")
    local = stage_path(stage, WORK_ROOT)
    if local.exists():
        unexpected = [
            p
            for p in local.rglob("*")
            if p.is_file()
            and p.relative_to(local).as_posix() not in manifest["files"]
            and p.name != "_COMPLETE.json"
        ]
        if unexpected:
            local.rename(local.with_name(f"{MODE}.incomplete-{time.time_ns()}"))
    local.mkdir(parents=True, exist_ok=True)
    names = manifest["files"] if wanted is None else wanted
    for name in names:
        require(name in manifest["files"], f"Artifact file not declared: {name}")
        info = manifest["files"][name]
        require(
            (remote / name).stat().st_size == info["bytes"], "Published size mismatch."
        )
        copy_verified(remote / name, local / name, info["sha256"])
    return local, manifest


def reusable(stage, parameters, inputs, catalog_id):
    marker = stage_path(stage, DRIVE_ROOT) / "_COMPLETE.json"
    if not marker.exists():
        return None
    path, manifest = load_stage(stage, catalog_id)
    require(
        manifest["model_parameters"] == parameters,
        "Existing parameters differ; version the run.",
    )
    require(
        manifest["input_fingerprints"] == inputs,
        "Existing inputs differ; version the run.",
    )
    return path, manifest


def runtime_versions():
    from importlib.metadata import PackageNotFoundError, version

    versions = {}
    for name in (
        "duckdb",
        "pyarrow",
        "numpy",
        "scipy",
        "implicit",
        "faiss-cpu",
        "torch",
        "transformers",
        "sentence-transformers",
        "scikit-learn",
        "lightgbm",
        "joblib",
    ):
        try:
            versions[name] = version(name)
        except PackageNotFoundError:
            continue
    return versions


def publish(stage, local, parameters, inputs, catalog_id, row_count, snapshot=None):
    local = Path(local)
    files = {}
    for path in sorted(local.rglob("*")):
        if (
            path.is_file()
            and path.name != "_COMPLETE.json"
            and not path.name.endswith(".partial")
        ):
            name = path.relative_to(local).as_posix()
            files[name] = {"bytes": path.stat().st_size, "sha256": sha256_file(path)}
    require(bool(files), "Refusing to publish an empty stage.")
    manifest = {
        "stage": stage,
        "artifact_version": ARTIFACT_VERSION,
        "protocol_version": PROTOCOL_VERSION,
        "mode": MODE,
        "input_paths": sorted(inputs),
        "input_fingerprints": inputs,
        "model_parameters": parameters,
        "random_seed": SEED,
        "row_count": int(row_count),
        "snapshot_or_anchor": str(snapshot or CUTOFF),
        "catalog_id": catalog_id,
        "feature_schema_version": FEATURE_SCHEMA_VERSION,
        "output_paths": sorted(files),
        "validation_status": "validated",
        "files": files,
        "runtime_versions": runtime_versions(),
    }
    remote = stage_path(stage, DRIVE_ROOT)
    existing = remote / "_COMPLETE.json"
    if existing.exists():
        require(
            json.loads(existing.read_text()) == manifest,
            "Refusing to replace a completed stage.",
        )
        return manifest
    disk_guard(
        remote, sum(x["bytes"] for x in files.values()), "Publish validated stage"
    )
    remote.mkdir(parents=True, exist_ok=True)
    for name, info in files.items():
        copy_verified(local / name, remote / name, info["sha256"])
    # Completion marker is published last, after every checksum is verified.
    marker = local / "_COMPLETE.json"
    marker.write_text(json.dumps(manifest, indent=2, sort_keys=True))
    copy_verified(marker, existing)
    return manifest


def new_stage(stage):
    path = stage_path(stage, WORK_ROOT)
    if path.exists() and any(path.iterdir()):
        path.rename(path.with_name(f"{MODE}.incomplete-{time.time_ns()}"))
    path.mkdir(parents=True, exist_ok=True)
    # Completion manifests govern reuse of incomplete local runs.
    return path


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, sort_keys=True, default=str))


def connect_db():
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    db = duckdb.connect()
    db.execute(f"SET memory_limit={sql_string(MEMORY_LIMIT)}")
    db.execute(f"SET threads={THREADS}")
    db.execute(f"SET temp_directory={sql_string(WORK_ROOT / 'spill')}")
    db.execute("SET preserve_insertion_order=false")
    return db


def export_query(db, query, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(".partial")
    db.execute(
        f"COPY ({query}) TO {sql_string(partial)} "
        "(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 65536)"
    )
    os.replace(partial, path)


def view_file(db, name, path):
    db.execute(
        f"CREATE OR REPLACE VIEW {name} AS "
        f"SELECT * FROM read_parquet({sql_string(path)})"
    )


def query_batches(db, anchor):
    previous = ""
    while True:
        rows = db.execute(
            (
                "SELECT query_id, customer_id, anchor_date, split FROM "
                "queries WHERE anchor_date=? AND eligible AND query_id>? "
                "ORDER BY query_id LIMIT ?"
            ),
            [anchor, previous, QUERY_BATCH_SIZE],
        ).fetchall()
        if not rows:
            return
        batch = [
            dict(
                zip(
                    ("query_id", "customer_id", "anchor_date", "split"),
                    row,
                    strict=True,
                )
            )
            for row in rows
        ]
        yield batch
        previous = batch[-1]["query_id"]


def attach_queries(db, batch):
    db.register("batch_queries", pa.Table.from_pylist(batch))


def history_for_batch(db, batch, anchor):
    attach_queries(db, batch)
    # Exact historical counts; only the query preference representation is capped.
    rows = db.execute(
        (
            "WITH grouped AS (\n"
            "        SELECT t.customer_id, t.article_id, count(*) AS n, "
            "max(t.t_dat) AS last_date\n"
            "        FROM transactions t JOIN batch_queries b "
            "USING(customer_id)\n"
            "        WHERE t.t_dat < ? GROUP BY "
            "t.customer_id,t.article_id)\n"
            "        SELECT * FROM grouped QUALIFY row_number() OVER\n"
            "        (PARTITION BY customer_id ORDER BY last_date DESC, "
            "article_id) <= ?\n"
            "        ORDER BY customer_id,last_date DESC,article_id"
        ),
        [anchor, MAX_HISTORY_ITEMS],
    ).fetchall()
    result = {}
    for customer, item, count, last in rows:
        weight = float(count) * math.exp(-(anchor - last).days / 30.0)
        result.setdefault(customer, []).append((item, weight))
    return result


def open_context():
    processed, pm = load_stage(
        "processed",
        wanted=[
            "transactions.parquet",
            "articles.parquet",
            "user_id_map.parquet",
            "article_id_map.parquet",
        ],
    )
    protocol, qm = load_stage("protocol", pm["catalog_id"])
    db = connect_db()
    for name in ("transactions", "articles", "user_id_map", "article_id_map"):
        view_file(db, name, processed / f"{name}.parquet")
    for name in ("eligible_catalog", "anchors"):
        view_file(db, name, protocol / f"{name}.parquet")
    view_file(db, "queries", protocol / "queries" / "*" / "queries.parquet")
    view_file(db, "labels", protocol / "labels" / "*" / "labels.parquet")
    catalog = [
        r[0]
        for r in db.execute(
            "SELECT article_id FROM eligible_catalog ORDER BY article_id"
        ).fetchall()
    ]
    return db, catalog, pm, qm


def runtime_start():
    require(MODE in ("smoke", "full"), "MODE must be smoke or full.")
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    np.random.seed(SEED)
    available = None
    memory_file = Path("/proc/meminfo")
    if memory_file.exists():
        for line in memory_file.read_text().splitlines():
            if line.startswith("MemAvailable:"):
                available = int(line.split()[1]) / 1024**2
        require(
            available is not None and available >= 6,
            (
                "At least 6 GiB available system RAM is required; no queries "
                "will be truncated."
            ),
        )
    print(
        {
            "mode": MODE,
            "result_scope": "diagnostic" if MODE == "smoke" else "full protocol",
            "local_free_GiB": round(shutil.disk_usage(WORK_ROOT).free / 1024**3, 2),
            "available_system_RAM_GiB": available,
            "duckdb_memory_limit": MEMORY_LIMIT,
            "threads": THREADS,
        }
    )

## Canonical conversion and chronological query construction
Source key/date/join failures stop the stage. Source lines are checked with multiset comparison; labels alone are deduplicated. Query rosters record eligibility and missing history. Exclusion counts are computed before smoke query limiting.

In [4]:
def prepare_canonical(db, sources, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    required = {
        "articles": {
            "article_id",
            "prod_name",
            "product_type_name",
            "product_group_name",
            "colour_group_name",
            "detail_desc",
        },
        "customers": {"customer_id", "age"},
        "transactions_train": {
            "t_dat",
            "customer_id",
            "article_id",
            "price",
            "sales_channel_id",
        },
    }
    for table, fields in required.items():
        db.execute(
            f"CREATE OR REPLACE VIEW raw_{table} AS SELECT * FROM read_csv_auto("
            f"{sql_string(sources[table])}, header=true, "
            "all_varchar=true, strict_mode=true)"
        )
        columns = {r[0] for r in db.execute(f"DESCRIBE raw_{table}").fetchall()}
        require(fields <= columns, f"{table}: required columns missing.")
    for table, key in [("articles", "article_id"), ("customers", "customer_id")]:
        n, distinct, invalid = db.execute(f"""SELECT count(*),count(DISTINCT {key}),
            count(*) FILTER (WHERE {key} IS NULL OR trim({key})='')
            FROM raw_{table}""").fetchone()
        require(n == distinct and invalid == 0, f"{table}: invalid or duplicate keys.")
    require(
        db.execute(
            "SELECT count(*) FROM raw_articles WHERE NOT "
            "regexp_full_match(article_id,'[0-9]{10}')"
        ).fetchone()[0]
        == 0,
        "Noncanonical article IDs; numeric coercion is not permitted.",
    )
    invalid = db.execute(
        "SELECT count(*) FROM raw_transactions_train t WHERE\n"
        "        try_cast(t_dat AS DATE) IS NULL OR NOT "
        "regexp_full_match(t_dat,'[0-9]{4}-[0-9]{2}-[0-9]{2}')\n"
        "        OR customer_id IS NULL OR article_id IS NULL OR\n"
        "        NOT regexp_full_match(article_id,'[0-9]{10}') OR\n"
        "        try_cast(price AS DOUBLE) IS NULL OR NOT "
        "isfinite(try_cast(price AS DOUBLE)) OR\n"
        "        try_cast(sales_channel_id AS INTEGER) IS NULL OR\n"
        "        NOT EXISTS (SELECT 1 FROM raw_articles a WHERE "
        "a.article_id=t.article_id) OR\n"
        "        NOT EXISTS (SELECT 1 FROM raw_customers c WHERE "
        "c.customer_id=t.customer_id)"
    ).fetchone()[0]
    require(invalid == 0, "Invalid transaction dates, prices, keys or catalog joins.")
    full_count = db.execute("SELECT count(*) FROM raw_transactions_train").fetchone()[0]
    filter_sql = ""
    if MODE == "smoke":
        db.execute(f"""CREATE TEMP TABLE selected_customers AS SELECT customer_id
            FROM raw_customers ORDER BY md5(customer_id || '{SEED}')
            LIMIT {SMOKE_CUSTOMERS}""")
        filter_sql = (
            " WHERE customer_id IN (SELECT customer_id FROM selected_customers)"
        )
    export_query(
        db,
        "SELECT * FROM raw_articles ORDER BY article_id",
        output / "articles.parquet",
    )
    export_query(
        db,
        ("SELECT * EXCLUDE(age),try_cast(age AS DOUBLE) AS age FROM raw_customers")
        + filter_sql
        + " ORDER BY customer_id",
        output / "customers.parquet",
    )
    export_query(
        db,
        (
            "SELECT cast(t_dat AS DATE) AS t_dat,customer_id,article_id,\n"
            "        cast(price AS DOUBLE) AS price,cast(sales_channel_id"
            " AS INTEGER) AS sales_channel_id\n"
            "        FROM raw_transactions_train"
        )
        + filter_sql
        + " ORDER BY t_dat,customer_id,article_id",
        output / "transactions.parquet",
    )
    for table in ("articles", "customers", "transactions"):
        view_file(db, table, output / f"{table}.parquet")
    for table, key, filename, idx in [
        ("customers", "customer_id", "user_id_map", "user_idx"),
        ("articles", "article_id", "article_id_map", "item_idx"),
    ]:
        export_query(
            db,
            f"SELECT {key},cast(row_number() OVER "
            f"(ORDER BY {key})-1 AS BIGINT) AS {idx} "
            f"FROM {table} ORDER BY {key}",
            output / f"{filename}.parquet",
        )
    selected_count = db.execute(
        "SELECT count(*) FROM raw_transactions_train" + filter_sql
    ).fetchone()[0]
    count = db.execute("SELECT count(*) FROM transactions").fetchone()[0]
    require(count == selected_count, "Purchase-line multiplicities changed.")
    # EXCEPT ALL checks multiplicities, not only unique rows.
    difference = db.execute(
        (
            "SELECT count(*) FROM (\n"
            "        (SELECT cast(t_dat AS "
            "DATE),customer_id,article_id,cast(price AS DOUBLE),\n"
            "         cast(sales_channel_id AS INTEGER) FROM "
            "raw_transactions_train"
        )
        + filter_sql
        + """)
        EXCEPT ALL SELECT * FROM transactions)"""
    ).fetchone()[0]
    require(difference == 0, "Canonical transaction values or multiplicities changed.")
    catalog = [
        r[0]
        for r in db.execute(
            "SELECT DISTINCT t.article_id FROM transactions t\n"
            "        JOIN articles a USING(article_id) WHERE t.t_dat < "
            "DATE '2020-07-28'\n"
            "        ORDER BY t.article_id"
        ).fetchall()
    ]
    require(bool(catalog), "No pre-cutoff eligible articles in this dataset/mode.")
    diagnostics = {
        "source_purchase_lines": full_count,
        "canonical_purchase_lines": count,
        "multiplicities_preserved": True,
        "mode": MODE,
        "smoke_customer_rule": f"md5(customer_id || {SEED}) first {SMOKE_CUSTOMERS}",
        "catalog_size": len(catalog),
    }
    write_json(output / "data_quality.json", diagnostics)
    return catalog, diagnostics


def prepare_protocol(db, catalog, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    db.register("catalog_input", pa.table({"article_id": catalog}))
    export_query(
        db,
        "SELECT * FROM catalog_input ORDER BY article_id",
        output / "eligible_catalog.parquet",
    )
    db.register("schedule_input", pa.Table.from_pylist(anchor_schedule()))
    export_query(
        db,
        "SELECT * FROM schedule_input ORDER BY anchor_date",
        output / "anchors.parquet",
    )
    diagnostics = []
    for anchor in anchor_schedule():
        a, end, split = anchor["anchor_date"], anchor["window_end"], anchor["split"]
        db.execute(
            (
                "CREATE OR REPLACE TEMP TABLE future AS\n"
                "            SELECT DISTINCT t.customer_id,t.article_id,\n"
                "              EXISTS(SELECT 1 FROM catalog_input c WHERE "
                "c.article_id=t.article_id) AS in_catalog\n"
                "            FROM transactions t WHERE t_dat>=? AND t_dat<?"
            ),
            [a, end],
        )
        db.execute(
            (
                "CREATE OR REPLACE TEMP TABLE query_roster AS\n"
                "            SELECT sha256(customer_id || '|' || cast(? AS "
                "VARCHAR)) AS query_id,\n"
                "              customer_id,cast(? AS DATE) AS "
                "anchor_date,cast(? AS VARCHAR) AS split,\n"
                "              bool_or(in_catalog) AS eligible,\n"
                "              NOT EXISTS(SELECT 1 FROM transactions h WHERE "
                "h.customer_id=f.customer_id\n"
                "                         AND h.t_dat < ?) AS missing_history"
                "\n"
                "            FROM future f GROUP BY customer_id"
            ),
            [a, a, split, a],
        )
        before = db.execute(
            "SELECT count(*),count(*) FILTER (WHERE NOT eligible),\n"
            "            count(*) FILTER (WHERE eligible AND "
            "missing_history) FROM query_roster"
        ).fetchone()
        excluded = db.execute(
            "SELECT count(*) FROM future WHERE NOT in_catalog"
        ).fetchone()[0]
        if MODE == "smoke":
            db.execute(f"""CREATE OR REPLACE TEMP TABLE query_roster AS
                SELECT * FROM query_roster
                ORDER BY eligible DESC,md5(query_id || '{SEED}')
                LIMIT {SMOKE_QUERIES_PER_ANCHOR}""")
        folder = f"{split}_{a.isoformat()}"
        export_query(
            db,
            "SELECT * FROM query_roster ORDER BY query_id",
            output / "queries" / folder / "queries.parquet",
        )
        export_query(
            db,
            (
                "SELECT "
                "q.query_id,q.customer_id,q.anchor_date,q.split,f.article_id\n"
                "            FROM query_roster q JOIN future f "
                "USING(customer_id)\n"
                "            WHERE q.eligible AND f.in_catalog ORDER BY "
                "q.query_id,f.article_id"
            ),
            output / "labels" / folder / "labels.parquet",
        )
        eligible_count = db.execute(
            "SELECT count(*) FROM query_roster WHERE eligible"
        ).fetchone()[0]
        require(
            eligible_count > 0,
            f"No eligible queries at {a}; enlarge smoke subset or check input.",
        )
        diagnostics.append(
            {
                "split": split,
                "anchor_date": str(a),
                "future_active_queries": before[0],
                "excluded_queries": before[1],
                "missing_history_queries": before[2],
                "excluded_unique_customer_article_positives": excluded,
                "retained_eligible_queries": eligible_count,
            }
        )
    manifest = {
        "protocol_version": PROTOCOL_VERSION,
        "mode": MODE,
        "seed": SEED,
        "catalog_id": identity(catalog),
        "retrieval_cutoff": str(CUTOFF),
        "anchors": anchor_schedule(),
        "historical_rule": "t_dat < anchor_date",
        "label_rule": "anchor_date <= t_dat < anchor_date + 7 days",
        "query_population": "future-purchase-active, at least one eligible positive",
        "diagnostics": diagnostics,
        "smoke_query_limit": SMOKE_QUERIES_PER_ANCHOR,
        "static_metadata_assumption": True,
    }
    write_json(output / "protocol_manifest.json", manifest)
    return manifest


def run_data_preparation():
    source_names = {
        "articles": "articles.csv",
        "customers": "customers.csv",
        "transactions_train": "transactions_train.csv",
    }
    sources, fingerprints = {}, {}
    for table, filename in source_names.items():
        source = DRIVE_ROOT / "original data" / filename
        require(
            source.exists(),
            f"Existing CSV missing: {source}; this notebook never downloads data.",
        )
        fingerprints[str(source)] = sha256_file(source)
        sources[table] = copy_verified(
            source, LOCAL_ROOT / filename, fingerprints[str(source)]
        )
    size = sum(p.stat().st_size for p in sources.values())
    disk_guard(WORK_ROOT, size * 3, "Canonical Parquet and DuckDB spill")
    params = {"mode": MODE, "smoke_customers": SMOKE_CUSTOMERS, "seed": SEED}
    marker = stage_path("processed", DRIVE_ROOT) / "_COMPLETE.json"
    db = connect_db()
    if marker.exists():
        path, pm = load_stage("processed")
        require(
            pm["input_fingerprints"] == fingerprints
            and pm["model_parameters"] == params,
            "Canonical inputs/configuration changed; use a new artifact version.",
        )
        for name in ("articles", "customers", "transactions"):
            view_file(db, name, path / f"{name}.parquet")
        catalog = [
            r[0]
            for r in db.execute(
                "SELECT DISTINCT article_id FROM transactions WHERE "
                "t_dat<DATE '2020-07-28' ORDER BY article_id"
            ).fetchall()
        ]
    else:
        path = new_stage("processed")
        catalog, quality = prepare_canonical(db, sources, path)
        pm = publish(
            "processed",
            path,
            params,
            fingerprints,
            identity(catalog),
            quality["canonical_purchase_lines"],
        )
    inputs = {"processed_manifest": identity(pm)}
    params = {
        "anchors": [str(x["anchor_date"]) for x in anchor_schedule()],
        "smoke_queries_per_anchor": SMOKE_QUERIES_PER_ANCHOR,
        "seed": SEED,
    }
    existing = reusable("protocol", params, inputs, identity(catalog))
    if existing is None:
        path = new_stage("protocol")
        protocol = prepare_protocol(db, catalog, path)
        total = sum(x["retained_eligible_queries"] for x in protocol["diagnostics"])
        publish("protocol", path, params, inputs, identity(catalog), total)
    metric_smoke_checks()
    db.close()
    print("Canonical data/protocol validated and published. No model fitted.")

## Execute this stage
This is the entry point. It requires the documented completed prerequisites. No models or datasets are downloaded during local source validation.

In [5]:
runtime_start()
metric_smoke_checks()
run_data_preparation()

Mounted at /content/drive
{'mode': 'full', 'result_scope': 'full protocol', 'local_free_GiB': 193.02, 'available_system_RAM_GiB': 49.37171173095703, 'duckdb_memory_limit': '4GB', 'threads': 2}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (12 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Canonical data/protocol validated and published. No model fitted.
